In [1]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv('ind_attribution.csv')
df.head()

,cookie,conversion,Facebook,Instagram,Online_Display,Online_Video,Paid_Search,days_since_first
0,00000FkCnDfDDf0iC97iC703B,0,0,1,1,0,0,25
1,0000nACkD9nFkBBDECD3ki00E,0,0,0,0,0,1,8
2,0003EfE37E93D0BC03iBhBBhF,0,0,0,0,0,1,3
3,0007iiAiFh3ifoo9Ehn3ABB0F,0,1,1,0,0,0,9
4,000A9AfDohfiBAFB0FDf3kDEE,0,0,0,0,1,0,5


# Estimate a Logistic regression model

In [9]:
import statsmodels.api as sm

X = df.drop(columns=['cookie', 'conversion']) 
y = df['conversion']

# Add constant for intercept
X_const = sm.add_constant(X)

# Fit logistic regression using statsmodels
logit_model = sm.Logit(y, X_const)
result = logit_model.fit()

# Display summary with p-values and fit metrics
print(result.summary())

Optimization terminated successfully.
         Current function value: 0.682668
         Iterations 4
                           Logit Regression Results                           
Dep. Variable:             conversion   No. Observations:                 2000
Model:                          Logit   Df Residuals:                     1993
Method:                           MLE   Df Model:                            6
Date:                Thu, 29 May 2025   Pseudo R-squ.:                 0.01512
Time:                        23:50:47   Log-Likelihood:                -1365.3
converged:                       True   LL-Null:                       -1386.3
Covariance Type:            nonrobust   LLR p-value:                 1.909e-07
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -0.2927      0.122     -2.402      0.016      -0.532      -0.054
Facebook   

# Get ready for attribution

## First, select only those with conversion =1

In [10]:
df_converted = df[df['conversion'] == 1]
df_converted.head()

,cookie,conversion,Facebook,Instagram,Online_Display,Online_Video,Paid_Search,days_since_first
16,000hCBnCB7oi7ADAEnEBCnBEE,1,0,1,0,1,0,17
20,000kiDB3D0fCfDAohCDB3ohko,1,1,0,0,0,0,10
25,0033CAi9nhFf0DAA7DDCC3FAn,1,0,0,0,1,0,17
94,009B3hhBfAfhDCEEFEnB9ABBB,1,0,0,0,0,1,0
130,00ADnBD9B979fE0fnoB9CA0E9,1,0,0,0,0,1,3


## Second, calculate the purchase probability when access each channel

In [15]:
channels = ['Facebook', 'Instagram', 'Online_Display', 'Online_Video', 'Paid_Search']
prob_by_channel = pd.DataFrame({'cookie': df_converted['cookie']})

# For each channel, set that channel to 1 and all others to 0, keep days_since_first as is
for channel in channels:
    X_temp = pd.DataFrame(0, index=df_converted.index, columns=channels)
    X_temp[channel] = 1
    X_temp['days_since_first'] = df_converted['days_since_first'].values
    X_temp_const = sm.add_constant(X_temp, has_constant='add')
    prob = result.predict(X_temp_const)
    prob_by_channel[channel + '_prob'] = prob

prob_by_channel.head()

,cookie,Facebook_prob,Instagram_prob,Online_Display_prob,Online_Video_prob,Paid_Search_prob
16,000hCBnCB7oi7ADAEnEBCnBEE,0.431995,0.414534,0.389779,0.495754,0.350192
20,000kiDB3D0fCfDAohCDB3ohko,0.466937,0.449180,0.423854,0.531032,0.382978
25,0033CAi9nhFf0DAA7DDCC3FAn,0.431995,0.414534,0.389779,0.495754,0.350192
94,009B3hhBfAfhDCEEFEnB9ABBB,0.517335,0.499458,0.473735,0.580810,0.431653
130,00ADnBD9B979fE0fnoB9CA0E9,0.502206,0.484327,0.458669,0.566002,0.416865


## Third, calculate the posterior probabilities

In [17]:
priors = np.array([0.3, 0.1, 0.1, 0.3, 0.2])
posterior_unnorm = prob_by_channel[[c + '_prob' for c in channels]].values * priors
posterior_unnorm_df = pd.DataFrame(
    posterior_unnorm,
    columns=[c + '_posterior_unnorm' for c in channels],
    index=prob_by_channel.index
)

posterior_sum = posterior_unnorm_df.sum(axis=1)
posterior_df = posterior_unnorm_df.div(posterior_sum, axis=0)
posterior_df.head()

,Facebook_posterior_unnorm,Instagram_posterior_unnorm,Online_Display_posterior_unnorm,Online_Video_posterior_unnorm,Paid_Search_posterior_unnorm
16,0.302239,0.096674,0.090901,0.346847,0.163338
20,0.302362,0.096954,0.091488,0.343866,0.165330
25,0.302239,0.096674,0.090901,0.346847,0.163338
94,0.302480,0.097342,0.092329,0.339593,0.168255
130,0.302452,0.097228,0.092077,0.340873,0.167370


In [18]:
avg_posteriors = posterior_df.mean()
print(avg_posteriors)

Facebook_posterior_unnorm          0.302374
Instagram_posterior_unnorm         0.097034
Online_Display_posterior_unnorm    0.091667
Online_Video_posterior_unnorm      0.342958
Paid_Search_posterior_unnorm       0.165967
dtype: float64
